In [ ]:
### Load libraries and set params
library(Matrix)
library(parallel)
library(Seurat)
library(gridExtra)
library(ggplot2)
library(future)
library(ggpubr)
library(viridis)
library(openxlsx)
library(viridis)
library(scran)
library(gghighlight)
library(dplyr)
library(org.Dm.eg.db)
library(ggExtra)
library(scDblFinder)
library(patchwork)
library(RhpcBLASctl)
blas_set_num_threads(18)
library(peakRAM)
options(device=pdf)
options(future.globals.maxSize = 214748364800)
library(future)
plan("multicore", workers = 18)
library(SeuratDisk)
library(reshape2)
library(tidyverse)
library(RColorBrewer)
library(zellkonverter)

### Set directories
mainDir <- "/data/ebaird/scRNAseq/SCENTINELsep24/"
repDir <- paste0(mainDir, "wildtype/")
figDir <- paste0(repDir, "figs/")
tabDir <- paste0(repDir, "tables/")
refsDir <- paste0(mainDir, "refs/")
wt_dataDir <- paste0("/data/ebaird/scRNAseq/public_datasets/")


dir.create(repDir, recursive = TRUE, showWarnings = FALSE)
dir.create(figDir, recursive = TRUE, showWarnings = FALSE)
dir.create(tabDir, recursive = TRUE, showWarnings = FALSE)

### Set colours
mycols <- c(1, '#ffffe5','#fff7bc','#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506')
mycols11 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray")
mycols13 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green")
mycols17 <- c(1, '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "gray", "blue", "green", rainbow(4))

mycols20 <- c("yellow", '#fee391','#fec44f','#fe9929','#ec7014','#cc4c02','#993404','#662506', "purple", "violet", "chartreuse", "blue", "green", rainbow(4), "darkslategray3", "darksalmon", "darkorchid4", "cyan")

corner <- function(x) x[1:5,1:5]
cols <- c(colorRamps::matlab.like2(20)[1:18], "deeppink2", "deeppink3", "deeppink4")

getdensity <- function(x, y, ...) {
      dens <- MASS::kde2d(x, y, ...)
      ix <- findInterval(x, dens$x)
      iy <- findInterval(y, dens$y)
      ii <- cbind(ix, iy)
      return(dens$z[ii])
}

In [ ]:
# Load data from rds object
# seu_marques <- readRDS(paste0(wt_dataDir, "Marques/CB.105H.rds"))
# Update seurat object if created with old version of seurat
# seu_marques <- UpdateSeuratObject(seu_marques)

seu_dillon <- readRDS(paste0(wt_dataDir, "Dillon/progenitors_labeled_noLQ.rds"))

# seu_nguyen <- readRDS(paste0(wt_dataDir, "Nguyen/GSE235231_Third_instar_larvae_VNC_Serpe.rds"))

In [ ]:
# Load data from h5ad object
seu_michki <- readH5AD(paste0(wt_dataDir, "Michki/GSE153723_typeII_OL_merged_cleaned.h5ad"))
assayNames(seu_michki) <- "counts"
assay(seu_michki, "logcounts") <- log1p(assay(seu_michki, "counts"))
seu_michki <- as.Seurat(seu_michki)

In [ ]:
DefaultAssay(seu_dillon) <- "RNA"
# DefaultAssay(seu_nguyen) <- "RNA"
# DefaultAssay(seu_michki) <- "originalexp"
# DefaultAssay(seu_marques) <- "RNA"

In [ ]:
# Create empty list
seu_list <- list()
# Split into different seurat objects based on orig.ident or other meta.data
# seu_list <- SplitObject(seu, split.by = "orig.ident")
# Add seu manually to list
# seu_list[["marques"]] <- seu_marques
seu_list[["dillon_progenitors_atlas"]] <- seu_dillon
# seu_list[["nguyen"]] <- seu_nguyen
# seu_list[["michki"]] <- seu_michki
names(seu_list)

In [ ]:
### Load top 10 markers for our clusters
top <- read.csv(paste0(mainDir, "QC_clustering/tables/top10Markers_merged_clusters.csv"))

our_clusters <- list()

for (i in unique(top$cluster)) {
  i_char <- as.character(i)
  our_clusters[[i_char]] <- top[top$cluster == i, "gene"]
}

In [ ]:
our_clusters

In [ ]:
for (obj_name in names(seu_list)) {
  obj <- seu_list[[obj_name]]
  obj_plots <- list()
  
  # Add module scores for each cluster
  for (i in names(our_clusters)) {
    obj <- AddModuleScore(obj, features = list(our_clusters[[i]]), name = i)
  }
  
  # Plotting the scores for each cluster
  for (i in names(our_clusters)) {
    p <- FeaturePlot(obj, features = paste0(i, "1"), order = TRUE) +
      scale_color_gradientn(colours = cols) +
      ggtitle(paste(obj_name, i, sep = " - "))
    obj_plots[[i]] <- p
  }
  
  # Save plots for this object
  dir.create(file.path(figDir, obj_name), recursive = TRUE, showWarnings = FALSE)
  jpeg(paste0(figDir, obj_name, "/signature_scores_", obj_name, ".jpg"), width = 5000, height = 10000, res = 300)
  grid.arrange(grobs = obj_plots, ncol = 3)
  dev.off()
  
  # Update the object in the list
  seu_list[[obj_name]] <- obj
}


In [ ]:
# Check specific gene of interest
gene_of_interest <- "Bre1"

seu_list <- list(
  "Marques" = seu_marques,
  "Dillon" = seu_dillon,
  "Nguyen" = seu_nguyen,
  "Michki" = seu_michki
)

# Check if gene is present in all datasets
for (obj_name in names(seu_list)) {
  obj <- seu_list[[obj_name]]
  if (!(gene_of_interest %in% rownames(obj))) {
    stop(paste(gene_of_interest, "not found in", obj_name))
  }
}

# Plotting
plots <- list()
for (obj_name in names(seu_list)) {
  obj <- seu_list[[obj_name]]
  p <- FeaturePlot(obj, features = gene_of_interest, order = TRUE) +
    scale_color_gradientn(colours = cols) +
    ggtitle(paste(obj_name, gene_of_interest, sep = " - "))
  plots[[obj_name]] <- p
}

jpeg(paste0(figDir, "featureplot_", gene_of_interest, ".jpg"), width = 4000, height = 3000, res = 300)
grid.arrange(grobs = plots, ncol = 2)
dev.off()